# 卖家第二步：关联销售明细与核对金额
第十四份 Notebook 的理解题已经核对。本课把卖家属性带入商品明细，检查主键、数量、金额。
这一份可从原始 CSV 独立运行，不依赖前一份 Notebook 的内存变量。

| 工作表 | 一行代表什么 | 主键 |
| --- | --- | --- |
| sellers_clean | 一个卖家的属性记录 | seller_id |
| items_clean | 订单中的一条商品明细 | order_id + order_item_id |
| items_with_sellers | 一条商品明细，附带卖家属性 | order_id + order_item_id |

**本课目标：关联后保留真实销售记录，数量和整数分金额保持一致。**
本课使用原始全部明细，没有按日期或订单状态筛选。


## 1. 读取原表，复用已有的金额清洗函数
邮编、ID 和原始金额先按字符串读取。
clean_order_items 是前面课程的公共函数，负责规范明细序号、从原始金额精确换成整数分、解析发货截止时间。


In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

project_dir = Path.cwd()
if project_dir.name == 'notebooks':
    project_dir = project_dir.parent
if not (project_dir / 'src').is_dir():
    raise ValueError('请从 olist-operations-analytics 项目目录打开本课')

data_dir = project_dir.parent / 'archive'
src_dir = str(project_dir / 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from items_cleaning import clean_order_items
# 客户和卖家共用已保存的官方州代码参考，导入时不需要联网。
from customers_cleaning import STATE_CODES

def read_source(filename):
    return pd.read_csv(
        data_dir / filename, dtype=str, encoding='utf-8-sig',
        keep_default_na=False, na_values=['']
    )

sellers_raw = read_source('olist_sellers_dataset.csv')
items_raw = read_source('olist_order_items_dataset.csv')

print('卖家表行数：', len(sellers_raw))
print('原始商品明细行数：', len(items_raw))


卖家表行数： 3095
原始商品明细行数： 112650


## 2. 重现卖家工作副本和商品明细规范
与上一课相同，地区去首尾空白、规范大小写，邮编保留字符串；原表保持原值。
卖家 ID 缺失、空白或重复时先停止，不自动去重。

四个地区标记用于保留待核查信息。地区检查与 ID 关联分别判断，不因为地区标记就删除销售明细。


In [2]:
seller_ids = sellers_raw['seller_id']
if (
    seller_ids.isna().any()
    or seller_ids.str.strip().eq('').any()
    or seller_ids.duplicated().any()
):
    raise ValueError('卖家主键异常，请先核查')

sellers_clean = sellers_raw.copy()
for col in ['seller_zip_code_prefix', 'seller_city', 'seller_state']:
    sellers_clean[col] = sellers_clean[col].str.strip()
sellers_clean['seller_city'] = sellers_clean['seller_city'].str.lower()
sellers_clean['seller_state'] = sellers_clean['seller_state'].str.upper()

sellers_clean['flag_zip_format_invalid'] = (
    ~sellers_clean['seller_zip_code_prefix'].str.fullmatch('[0-9]{5}', na=False)
)
sellers_clean['flag_state_format_invalid'] = (
    ~sellers_clean['seller_state'].str.fullmatch('[A-Z]{2}', na=False)
)
sellers_clean['flag_state_unknown'] = ~sellers_clean['seller_state'].isin(STATE_CODES)
sellers_clean['flag_city_missing'] = (
    sellers_clean['seller_city'].isna() | sellers_clean['seller_city'].eq('')
)

# 返回完整明细副本，继续保留订单ID、明细序号、商品和卖家ID。
# 金额精确转换沿用前面验证过的函数，不先转 float 再乘100。
items_clean = clean_order_items(items_raw)

print('规范后商品明细行数：', len(items_clean))
display(items_clean[['price_cents', 'freight_cents']].dtypes)


规范后商品明细行数： 112650


price_cents      int64
freight_cents    int64
dtype: object

## 3. 先检查覆盖，再多对一左连接
左侧同一 seller_id 可以出现多次，右侧每个 seller_id 只对应一条卖家属性记录。
many_to_one 验证右侧键唯一，防止一条明细被匹配成多行；它不会替我们去重。

indicator='sellers_match' 增加匹配状态列。both 表示找到卖家记录，不代表地区信息已完成地理核查。
左连接保留明细作为主体；本课若存在未匹配卖家，先停止并核查。


In [3]:
# 检查方向：销售明细中的卖家 ID，能否在卖家属性表里找到。
# clean_order_items 已检查明细的 seller_id 非空。
unmatched_item_lines = (
    ~items_clean['seller_id'].isin(sellers_clean['seller_id'])
).sum()
print('找不到卖家属性的明细行数：', unmatched_item_lines)

if unmatched_item_lines > 0:
    raise ValueError('存在找不到卖家属性的销售明细，请先核查')

items_with_sellers = items_clean.merge(
    sellers_clean,
    on='seller_id',
    how='left',
    validate='many_to_one',
    indicator='sellers_match'
)

# 关联后继续检查明细组合主键；不能要求 seller_id 唯一。
item_key_cols = ['order_id', 'order_item_id']
key_duplicates = items_with_sellers.duplicated(item_key_cols).sum()
if key_duplicates > 0:
    raise ValueError('关联后商品明细组合主键重复，请核查')

print('关联前明细行数：', len(items_clean))
print('关联后明细行数：', len(items_with_sellers))
print('关联后明细组合主键重复数：', key_duplicates)
display(items_with_sellers['sellers_match'].value_counts())


找不到卖家属性的明细行数： 0
关联前明细行数： 112650
关联后明细行数： 112650
关联后明细组合主键重复数： 0


sellers_match
both          112650
left_only          0
right_only         0
Name: count, dtype: int64

## 4. 控制总额：数量、商品金额、运费
关联卖家属性不应改变销售事实。三行分别核对明细数量、商品金额、运费。
金额继续按整数分比较，避免浮点显示影响判断。

当前检查的是同一来源销售明细在关联前后的守恒，差额必须为零。
这里的金额是全量明细金额，不能直接称为已交付成交额或平台收入。


In [4]:
money_cols = ['price_cents', 'freight_cents']

# sum 默认会跳过缺失，核对前先确认两边的金额都完整。
if (
    items_clean[money_cols].isna().any().any()
    or items_with_sellers[money_cols].isna().any().any()
):
    raise ValueError('金额存在缺失，请先核查，不能直接求和')

seller_join_check = pd.DataFrame({
    '关联前': [
        len(items_clean),
        items_clean['price_cents'].sum(),
        items_clean['freight_cents'].sum()
    ],
    '关联后': [
        len(items_with_sellers),
        items_with_sellers['price_cents'].sum(),
        items_with_sellers['freight_cents'].sum()
    ]
}, index=['商品明细行数', '商品金额（分）', '运费（分）'])

seller_join_check['差额'] = (
    seller_join_check['关联后'] - seller_join_check['关联前']
)

if seller_join_check['差额'].ne(0).any():
    raise ValueError('关联前后数量或金额不一致，请先核查')

display(seller_join_check)


,关联前,关联后,差额
商品明细行数,112650,112650,0
商品金额（分）,1359164370,1359164370,0
运费（分）,225190954,225190954,0


## 5. 看一眼关联后的结果
下面仅筛选一个卖家的明细，查看补入的属性，没有按卖家聚合。
这个小表的每行仍有订单 ID 和明细序号，不是一行一个卖家。

下一步才学习按卖家统计，并区分明细行数、不同订单数、不同商品数。


In [5]:
# value_counts 统计卖家在明细中出现的行数，不是卖家订单数。
seller_item_line_counts = items_with_sellers['seller_id'].value_counts()
example_seller_id = seller_item_line_counts.index[0]

example_seller_items = items_with_sellers.loc[
    items_with_sellers['seller_id'].eq(example_seller_id)
]

display(example_seller_items[[
    'order_id', 'order_item_id', 'product_id', 'seller_id',
    'price_cents', 'freight_cents',
    'seller_zip_code_prefix', 'seller_city', 'seller_state'
]].head())

print('该卖家的明细行数：', len(example_seller_items))
print('该卖家涉及不同订单数：', example_seller_items['order_id'].nunique())


,order_id,order_item_id,product_id,seller_id,price_cents,freight_cents,seller_zip_code_prefix,seller_city,seller_state
23,001021efaa8636c29475e7734483457d,1,5d7c23067ed3fc8c6e699b9373d5890b,6560211a19b47992c3666cc44a7e94c0,4900,1510,05849,sao paulo,SP
117,00418a49a685c6bb60633291c3fab17a,1,5e21d5cab5d33e770d8150a4ee6117db,6560211a19b47992c3666cc44a7e94c0,4900,778,05849,sao paulo,SP
148,005a1dded353107dbabf8ffc83a20365,1,7e6c4a0bf900e259f50ba63331fd2785,6560211a19b47992c3666cc44a7e94c0,7500,1428,05849,sao paulo,SP
175,006e05e81c068ddc38e48ebe189b922e,1,02475368dfb38934fe55f574024fe1d7,6560211a19b47992c3666cc44a7e94c0,2900,1410,05849,sao paulo,SP
182,00750f68bb64537034e93554c64b41b5,1,526f833e1eb0a32ad9351145a761b9ab,6560211a19b47992c3666cc44a7e94c0,12900,816,05849,sao paulo,SP


该卖家的明细行数： 2033
该卖家涉及不同订单数： 1854


## 我的理解
先运行完本课，确认匹配状态、主键和控制表，再补充：

1. items_with_sellers 的一行代表什么，主键是什么？它现在是一行一个卖家的表吗？
一笔订单中的一条商品明细，包括有order_id,order_item_id,product_id,sell_id等，主键是order_id+order_item_Id,不是一行一个卖家。
2. 为什么增加卖家属性后，商品金额和运费的控制差额必须是 0？如果不为 0，是否应该直接改金额让它相等？
因为表只是增加了卖家属性，其他数据均没有改变，不为0不能该金额。
3. 为什么同一卖家的明细行数，可能大于它涉及的不同订单数？
同一订单中，如果多条商品明细属于同一个卖家，该卖家的明细数就会大于不同订单数。

本课只在内存中检查，不覆盖原始文件和前面的 Notebook。后续将流程整理为公共函数、独立导出和阶段报告。
